# Experiment 2.0 — xP-Excluded Linear Hurdle Baseline

Reproduces Phase 1's winning hurdle architecture (LR Stage 1 + Ridge Stage 2) with `xP` removed from both stages. This is the bar Experiment 2.1's MLP must beat.

**Numbers that gate the 2.1 architecture decision:**

- **Stage-1 AUC without xP** — controls whether the hurdle structure is justified for 2.1:
  - `≥ 0.93` → hurdle justified
  - `0.90 – 0.93` → borderline; single-stage MLP worth a sanity check
  - `< 0.90` → reconsider hurdle
- **Combined val R² and MAE** — the linear baseline the MLP must beat
- **Sign of `total_points_roll3` in the no-xP Lasso** — sanity-check that removing xP redistributes signal as Phase 1 predicted

**Splits:** train = 2021-22 + 2022-23, val = 2023-24 GW 2–33. Test untouched in Phase 2.


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression, RidgeCV, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("./data/processed/fpl_modeling_data.csv")


## 1. Load data and define splits

In [3]:
df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows × {df.shape[1]} cols")

TRAIN_SEASONS = ["2021-22", "2022-23"]
HOLDOUT_SEASON = "2023-24"
TEST_GW_START = 34  # test set untouched in Phase 2

train_mask = df["season"].isin(TRAIN_SEASONS)
val_mask = (df["season"] == HOLDOUT_SEASON) & (df["GW"] < TEST_GW_START)

print(f"Train: {train_mask.sum():,} rows  ({TRAIN_SEASONS})")
print(f"Val:   {val_mask.sum():,} rows  ({HOLDOUT_SEASON} GW 2–{TEST_GW_START - 1})")


Loaded 75,075 rows × 56 cols
Train: 46,991 rows  (['2021-22', '2022-23'])
Val:   23,825 rows  (2023-24 GW 2–33)


## 2. Build feature lists defensively

Numeric columns minus identifiers, target, stage-1 labels, and `cluster_id`. Surface excluded columns so any undocumented Phase 0 output is visible.

**Exclusions beyond `played_any`:** `played_60min` is the other stage-1 label from Phase 0 — using it as a feature would leak the play decision in a different form. `cluster_id` was locked-out in Phase 1's ablation 1.1 (cost val R²). Override here if you want either back in.


In [4]:
IDENTIFIERS = ["name", "name_key", "season", "GW", "player_season", "team", "position"]
TARGET = "total_points"
STAGE1_LABELS = ["played_any", "played_60min"]
DROPPED_BY_PHASE1 = ["cluster_id"]

EXCLUDE = set(IDENTIFIERS + [TARGET] + STAGE1_LABELS + DROPPED_BY_PHASE1)

assert "played_any" in df.columns, "played_any missing — Phase 0 output incomplete?"

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
non_numeric = [c for c in df.columns if c not in numeric_cols]

features_with_xp = [c for c in numeric_cols if c not in EXCLUDE]
features_no_xp   = [c for c in features_with_xp if c != "xP"]

print(f"Features (with xP):    {len(features_with_xp)}")
print(f"Features (without xP): {len(features_no_xp)}")

print(f"\nExcluded non-numeric columns:")
for c in non_numeric:
    print(f"  {c}")

print(f"\nExcluded numeric columns (IDs / target / labels / dropped):")
for c in sorted(EXCLUDE & set(numeric_cols)):
    print(f"  {c}")


Features (with xP):    44
Features (without xP): 43

Excluded non-numeric columns:
  name_key
  season
  name
  team
  position
  was_home
  player_season

Excluded numeric columns (IDs / target / labels / dropped):
  GW
  cluster_id
  played_60min
  played_any
  total_points


## 3. Stage 1 — Logistic Regression with and without xP

Fit on the full train set. The no-xP version is the actual stage 1 used downstream; the with-xP version is run only to measure the AUC contribution from `xP` directly, gating the 2.1 architecture choice.


In [5]:
def fit_stage1_lr(features, df, train_mask, val_mask, label="played_any"):
    X_train = df.loc[train_mask, features].values
    y_train = df.loc[train_mask, label].values
    X_val   = df.loc[val_mask,   features].values
    y_val   = df.loc[val_mask,   label].values

    scaler = StandardScaler().fit(X_train)
    lr = LogisticRegression(max_iter=2000, random_state=SEED).fit(
        scaler.transform(X_train), y_train
    )
    p_val = lr.predict_proba(scaler.transform(X_val))[:, 1]
    return p_val, roc_auc_score(y_val, p_val), lr, scaler


p_play_with_xp, auc_with_xp, _, _                   = fit_stage1_lr(features_with_xp, df, train_mask, val_mask)
p_play_no_xp,   auc_no_xp,   lr_s1_no_xp, scaler_s1 = fit_stage1_lr(features_no_xp,   df, train_mask, val_mask)

gap = auc_with_xp - auc_no_xp
print(f"Stage-1 AUC WITH xP:    {auc_with_xp:.4f}")
print(f"Stage-1 AUC WITHOUT xP: {auc_no_xp:.4f}")
print(f"Gap (xP contribution):  {gap:+.4f}")

if auc_no_xp >= 0.93:
    verdict = "hurdle structure justified for 2.1"
elif auc_no_xp >= 0.90:
    verdict = "borderline — sanity-check single-stage MLP for 2.1"
else:
    verdict = "reconsider hurdle structure for 2.1"
print(f"\nVERDICT (no-xP AUC = {auc_no_xp:.4f}): {verdict}")


Stage-1 AUC WITH xP:    0.9468
Stage-1 AUC WITHOUT xP: 0.9300
Gap (xP contribution):  +0.0168

VERDICT (no-xP AUC = 0.9300): hurdle structure justified for 2.1


## 4. Stage 2 — RidgeCV on played-only rows (no xP)

Own scaler fit on the played-only train subset; applied to all val rows for the combined prediction below.


In [6]:
played_train_mask = train_mask & (df["played_any"] == 1)

X_train_s2 = df.loc[played_train_mask, features_no_xp].values
y_train_s2 = df.loc[played_train_mask, "total_points"].values
X_val_s2   = df.loc[val_mask,           features_no_xp].values

scaler_s2 = StandardScaler().fit(X_train_s2)
ridge = RidgeCV(alphas=np.logspace(-3, 3, 13)).fit(
    scaler_s2.transform(X_train_s2), y_train_s2
)
cond_points_val = ridge.predict(scaler_s2.transform(X_val_s2))

print(f"Stage 2 trained on {played_train_mask.sum():,} played-only train rows")
print(f"RidgeCV selected alpha: {ridge.alpha_}")


Stage 2 trained on 20,022 played-only train rows
RidgeCV selected alpha: 316.22776601683796


## 5. Combine — ŷ = P(play) · E[points | played]

No thresholding on stage 1. Multiply probabilities directly to preserve calibration on borderline rotation cases.


In [7]:
y_pred = p_play_no_xp * cond_points_val
y_val_true = df.loc[val_mask, "total_points"].values

r2  = r2_score(y_val_true, y_pred)
mae = mean_absolute_error(y_val_true, y_pred)

print(f"Hurdle (no xP) val R²:  {r2:.4f}")
print(f"Hurdle (no xP) val MAE: {mae:.4f}")
print(f"\nReference:")
print(f"  xP passthrough:         R² 0.5169, MAE 0.8129")
print(f"  Phase 1 hurdle WITH xP: R² 0.6386, MAE 0.7116")


Hurdle (no xP) val R²:  0.2999
Hurdle (no xP) val MAE: 0.9762

Reference:
  xP passthrough:         R² 0.5169, MAE 0.8129
  Phase 1 hurdle WITH xP: R² 0.6386, MAE 0.7116


## 6. Per-position MAE breakdown

In [8]:
val_df = df.loc[val_mask].assign(
    y_pred=y_pred,
    abs_err=lambda d: np.abs(d["total_points"] - d["y_pred"]),
)
per_pos = (
    val_df.groupby("position")
          .agg(n=("abs_err", "size"), mae=("abs_err", "mean"))
          .reindex(["GK", "DEF", "MID", "FWD"])
          .round(4)
)
print(per_pos.to_string())


              n     mae
position               
GK         2736  0.6188
DEF        7710  1.0641
MID       10300  0.9880
FWD        3079  1.0340


## 7. LassoCV coefficient comparison

Both fits run on the played-only train subset. Phase 1 predictions to verify:

- **(a)** standardized `xP` coefficient ≈ 2.4 in the with-xP fit
- **(b)** `total_points_roll3` flips back to **positive** in the without-xP fit

If `LassoCV` selects an alpha far from Phase 1's reference of `~1e-4`, the magnitude in (a) may differ — the structural finding is the sign table below.


In [9]:
def fit_lasso(features, df, played_train_mask):
    X = df.loc[played_train_mask, features].values
    y = df.loc[played_train_mask, "total_points"].values
    scaler = StandardScaler().fit(X)
    lasso = LassoCV(cv=5, random_state=SEED, max_iter=20000).fit(scaler.transform(X), y)
    return pd.Series(lasso.coef_, index=features), lasso.alpha_


coef_with_xp, alpha_with_xp = fit_lasso(features_with_xp, df, played_train_mask)
coef_no_xp,   alpha_no_xp   = fit_lasso(features_no_xp,   df, played_train_mask)

print(f"LassoCV alpha (with xP):    {alpha_with_xp:.6f}")
print(f"LassoCV alpha (without xP): {alpha_no_xp:.6f}")

print("\nTop 10 |coef| WITH xP:")
print(coef_with_xp.reindex(coef_with_xp.abs().sort_values(ascending=False).index)
                  .head(10).round(4).to_string())

print("\nTop 10 |coef| WITHOUT xP:")
print(coef_no_xp.reindex(coef_no_xp.abs().sort_values(ascending=False).index)
                .head(10).round(4).to_string())

print("\nSign-check table:")
check_features = [
    "xP",
    "total_points_roll3", "total_points_lag1",
    "bps_roll3", "bps_lag1",
    "minutes_roll3", "minutes_lag1",
    "ict_index_roll3",
]
sign_check = pd.DataFrame({
    "with_xP":    [coef_with_xp.get(f, np.nan) for f in check_features],
    "without_xP": [coef_no_xp.get(f, np.nan)   for f in check_features],
}, index=check_features).round(4)
print(sign_check.to_string())


LassoCV alpha (with xP):    0.002996
LassoCV alpha (without xP): 0.007356

Top 10 |coef| WITH xP:
xP                    2.9949
total_points_roll3   -1.3928
is_dgw               -0.3607
total_points_lag1    -0.2337
minutes_roll3         0.2296
n_fixtures            0.2218
bps_roll3            -0.1784
goals_scored_roll3    0.1662
gws_played            0.1354
influence_roll3      -0.0971

Top 10 |coef| WITHOUT xP:
is_dgw                  0.5409
value                   0.5280
minutes_roll3           0.3140
pos_GK                  0.2768
transfers_in            0.1790
pos_DEF                 0.1527
selected                0.1204
minutes_lag1            0.1163
goals_conceded_roll3   -0.0961
influence_roll3        -0.0953

Sign-check table:
                    with_xP  without_xP
xP                   2.9949         NaN
total_points_roll3  -1.3928     -0.0000
total_points_lag1   -0.2337      0.0000
bps_roll3           -0.1784      0.0000
bps_lag1            -0.0283      0.0000
minutes_roll3   

## 8. Phase 2 summary table

In [10]:
summary = pd.DataFrame([
    {"Model": "xP passthrough (Phase 1 reference)",     "Val R²": 0.5169,       "Val MAE": 0.8129},
    {"Model": "Phase 1 hurdle WITH xP (ceiling)",       "Val R²": 0.6386,       "Val MAE": 0.7116},
    {"Model": "Phase 1 hurdle WITHOUT xP (Exp 2.0)",    "Val R²": round(r2, 4), "Val MAE": round(mae, 4)},
    {"Model": "Hurdle MLP, xP-excluded (Exp 2.1, TBD)", "Val R²": np.nan,       "Val MAE": np.nan},
])
print(summary.to_string(index=False))


                                 Model  Val R²  Val MAE
    xP passthrough (Phase 1 reference)  0.5169   0.8129
      Phase 1 hurdle WITH xP (ceiling)  0.6386   0.7116
   Phase 1 hurdle WITHOUT xP (Exp 2.0)  0.2999   0.9762
Hurdle MLP, xP-excluded (Exp 2.1, TBD)     NaN      NaN


In [12]:
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression, Ridge, Lasso
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, mean_absolute_error, r2_score

SEED = 42
np.random.seed(SEED)

DATA_PATH = Path("./data/processed/fpl_modeling_data.csv")
df = pd.read_csv(DATA_PATH)

TRAIN_SEASONS = ["2021-22", "2022-23"]
HOLDOUT_SEASON = "2023-24"
TEST_GW_START = 34

train_mask       = df["season"].isin(TRAIN_SEASONS)
val_mask         = (df["season"] == HOLDOUT_SEASON) & (df["GW"] < TEST_GW_START)
played_train_mask = train_mask & (df["played_any"] == 1)
played_val_mask   = val_mask   & (df["played_any"] == 1)

EXCLUDE = {"name", "name_key", "season", "GW", "player_season", "team", "position",
           "total_points", "played_any", "played_60min", "cluster_id"}
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
features_no_xp = [c for c in numeric_cols if c not in EXCLUDE and c != "xP"]

print(f"train: {train_mask.sum():,}  val: {val_mask.sum():,}  "
      f"played-only train: {played_train_mask.sum():,}  played-only val: {played_val_mask.sum():,}")
print(f"features (no xP): {len(features_no_xp)}")


train: 46,991  val: 23,825  played-only train: 20,022  played-only val: 9,257
features (no xP): 43


In [13]:
# Stage 1 LR (no xP) — same as Experiment 2.0
X_train_s1 = df.loc[train_mask, features_no_xp].values
y_train_s1 = df.loc[train_mask, "played_any"].values
X_val_s1   = df.loc[val_mask,   features_no_xp].values
y_val_s1   = df.loc[val_mask,   "played_any"].values

scaler_s1 = StandardScaler().fit(X_train_s1)
lr_s1 = LogisticRegression(max_iter=2000, random_state=SEED).fit(scaler_s1.transform(X_train_s1), y_train_s1)
p_play_no_xp = lr_s1.predict_proba(scaler_s1.transform(X_val_s1))[:, 1]

# Stage 2 setup — fit own scaler on played-only train
X_train_s2_raw       = df.loc[played_train_mask, features_no_xp].values
y_train_s2           = df.loc[played_train_mask, "total_points"].values
X_val_full_raw       = df.loc[val_mask,           features_no_xp].values
X_val_played_raw     = df.loc[played_val_mask,    features_no_xp].values

scaler_s2 = StandardScaler().fit(X_train_s2_raw)
X_train_s2     = scaler_s2.transform(X_train_s2_raw)
X_val_full     = scaler_s2.transform(X_val_full_raw)
X_val_played   = scaler_s2.transform(X_val_played_raw)

y_val_full   = df.loc[val_mask,        "total_points"].values
y_val_played = df.loc[played_val_mask, "total_points"].values

# Run Ridge at both alphas
rows = []
for alpha in [1.0, 316.2278]:
    ridge = Ridge(alpha=alpha, random_state=SEED).fit(X_train_s2, y_train_s2)

    # Stage 2 alone — predict E[points | played] on played-only val rows
    pred_played = ridge.predict(X_val_played)
    r2_s2  = r2_score(y_val_played, pred_played)
    mae_s2 = mean_absolute_error(y_val_played, pred_played)

    # Combined hurdle — P(play) * E[points | played] on full val
    cond_full = ridge.predict(X_val_full)
    pred_comb = p_play_no_xp * cond_full
    r2_c  = r2_score(y_val_full, pred_comb)
    mae_c = mean_absolute_error(y_val_full, pred_comb)

    rows.append({
        "alpha": alpha,
        "stage2_alone_R²":  round(r2_s2, 4),
        "stage2_alone_MAE": round(mae_s2, 4),
        "combined_R²":      round(r2_c, 4),
        "combined_MAE":     round(mae_c, 4),
    })

results = pd.DataFrame(rows)
print(results.to_string(index=False))

print("\nReference (from Experiment 2.0): combined R² 0.2999, combined MAE 0.9762 at α≈316")


   alpha  stage2_alone_R²  stage2_alone_MAE  combined_R²  combined_MAE
  1.0000           0.0852            2.0335       0.2992        0.9757
316.2278           0.0865            2.0357       0.2999        0.9762

Reference (from Experiment 2.0): combined R² 0.2999, combined MAE 0.9762 at α≈316


In [14]:
# Reuse scaler_s2 — same training subset and feature set as the LassoCV run in Experiment 2.0,
# so coefficients are directly comparable in standardized units.
lasso_fixed = Lasso(alpha=1e-4, max_iter=50000, random_state=SEED).fit(X_train_s2, y_train_s2)
coef_fixed = pd.Series(lasso_fixed.coef_, index=features_no_xp)

print(f"Nonzero coefficients: {(coef_fixed != 0).sum()}/{len(coef_fixed)}")

print("\nTop 10 |coef| at fixed α=1e-4 (no xP):")
print(coef_fixed.reindex(coef_fixed.abs().sort_values(ascending=False).index)
                .head(10).round(4).to_string())

check_features = [
    "total_points_roll3", "total_points_lag1",
    "bps_roll3", "bps_lag1",
    "minutes_roll3", "minutes_lag1",
    "ict_index_roll3",
]
print("\nSign check at α=1e-4 (compare to LassoCV α≈0.0074 from Experiment 2.0):")
for f in check_features:
    val = coef_fixed.get(f, np.nan)
    sign = "POSITIVE" if val > 1e-6 else ("NEGATIVE" if val < -1e-6 else "ZERO")
    print(f"  {f:<22s} {val:+.4f}  ({sign})")


Nonzero coefficients: 38/43

Top 10 |coef| at fixed α=1e-4 (no xP):
is_dgw                  0.5582
value                   0.5378
minutes_roll3           0.3834
pos_GK                  0.3083
influence_roll3        -0.2273
pos_DEF                 0.1756
transfers_in            0.1732
goals_conceded_roll3   -0.1603
bps_roll3               0.1432
minutes_lag1            0.1282

Sign check at α=1e-4 (compare to LassoCV α≈0.0074 from Experiment 2.0):
  total_points_roll3     -0.1112  (NEGATIVE)
  total_points_lag1      -0.0506  (NEGATIVE)
  bps_roll3              +0.1432  (POSITIVE)
  bps_lag1               -0.0218  (NEGATIVE)
  minutes_roll3          +0.3834  (POSITIVE)
  minutes_lag1           +0.1282  (POSITIVE)
  ict_index_roll3        +0.0000  (ZERO)
